# **MÓDULO 35 - Cross Validation**

Nesta tarefa, você trabalhará com uma base de dados que contém informações sobre variáveis ambientais coletadas para a detecção de incêndios. O objetivo é utilizar técnicas de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação na previsão da ocorrência de um incêndio com base nas variáveis fornecidas.


Descrição da Base de Dados
A base de dados contém as seguintes variáveis:

Unnamed:0: Índice (não é uma variável útil para o modelo)

UTC: Tempo em Segundos UTC

Temperature[C]: Temperatura do Ar (em graus Celsius)

Humidity[%]: Umidade do Ar (em porcentagem)

TVOC[ppb]: Total de Compostos Orgânicos Voláteis (medido em partes por bilhão)

eCO2[ppm]: Concentração equivalente de CO2 (medido em partes por milhão)

Raw H2: Hidrogênio molecular bruto, não compensado

Raw Ethanol: Etanol gasoso bruto

Pressure[hPA]: Pressão do Ar (em hectopascais)

PM1.0: Material particulado de tamanho < 1,0 µm

PM2.5: Material particulado de tamanho >1,0 µm e < 2,5 µm

NC0.5: Concentração numérica de material particulado de tamanho < 0,5 µm

NC1.0: Concentração numérica de material particulado de tamanho 0,5 µm < 1,0 µm

NC2.5: Concentração numérica de material particulado de tamanho 1,0 µm < 2,5 µm

CNT: Contador de amostras


E a variável alvo:

Fire Alarm: Indicador binário de incêndio (1 se houver incêndio, 0 caso contrário)

O objetivo desta tarefa é aplicar a técnica de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação. A validação cruzada ajudará a garantir que o modelo seja avaliado de maneira robusta e generalize bem para dados não vistos.

In [2]:
import pandas as pd
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# 1 - Carregue a base de dados, verifique os tipos de dados e também se há presença de dados faltantes ou nulos.

In [10]:
df_imoveis = pd.read_csv("Cientista de dados M35 - smoke_detection_iot.csv", delimiter=',')

df_imoveis

# 1) Visão geral 
df_imoveis.info()        
print(df_imoveis.dtypes) 

# 2) Contagem e percentual de valores nulos por coluna
missing = df_imoveis.isnull().sum()
missing_pct = (missing / len(df_imoveis)) * 100

missing_df = pd.concat([missing, missing_pct], axis=1)
missing_df.columns = ['missing_count', 'missing_pct']

# Mostrar apenas colunas com ao menos 1 valor nulo, ordenado por quantidade
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_count', ascending=False)
print(missing_df) 
 
# 3) confirmar ausência total de nulos
total_nulos = df_imoveis.isnull().sum().sum()
print("Total de valores nulos no dataframe:", total_nulos)
assert total_nulos == 0, "Existem valores nulos no dataframe (verifique missing)."
print("OK — não há valores nulos.")

# 4) verificar duplicatas
n_duplicadas = df_imoveis.duplicated().sum()
print("Número de linhas duplicadas:", n_duplicadas)
if n_duplicadas > 0:
    display(df_imoveis[df_imoveis.duplicated(keep=False)].head())

# 5) balanceamento da variável alvo 'Fire Alarm'
print("Contagem absoluta (Fire Alarm):")
print(df_imoveis['Fire Alarm'].value_counts().sort_index())
print("\nPercentual (Fire Alarm):")
print(df_imoveis['Fire Alarm'].value_counts(normalize=True).sort_index())

# 6) converter UTCpara datetime com timezone America/Sao_Paulo
df_imoveis['datetime_utc'] = pd.to_datetime(df_imoveis['UTC'], unit='s', utc=True)
df_imoveis['datetime_sp'] = df_imoveis['datetime_utc'].dt.tz_convert('America/Sao_Paulo')
# mostrar
df_imoveis[['UTC','datetime_utc','datetime_sp']].head()

# 7) estatísticas descritivas detalhadas por coluna numérica
display(df_imoveis.describe().T)

# 8) checar valores negativos em colunas que não deveriam ter (<0)
cols_verificar = [
    'TVOC[ppb]','eCO2[ppm]','Raw H2','Raw Ethanol',
    'Pressure[hPa]','PM1.0','PM2.5','NC0.5','NC1.0','NC2.5','CNT'
]
cols_existentes = [c for c in cols_verificar if c in df_imoveis.columns]
neg_counts = {c: int((df_imoveis[c] < 0).sum()) for c in cols_existentes}
print("Contagem de valores negativos por coluna (se houver):")
print(neg_counts)

# 9) detecção rápida de outliers por IQR 
import numpy as np
numeric_cols = df_imoveis.select_dtypes(include=[np.number]).columns.tolist()
# excluir a variável alvo das features
if 'Fire Alarm' in numeric_cols:
    numeric_cols.remove('Fire Alarm')

def count_iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return int(((series < lower) | (series > upper)).sum())

iqr_outliers = {c: count_iqr_outliers(df_imoveis[c]) for c in numeric_cols}
# mostrar ordenado pelas maiores contagens de outliers
import pandas as pd
pd.Series(iqr_outliers).sort_values(ascending=False)

# 10) correlação das features numéricas com 'Fire Alarm'
if 'Fire Alarm' in df_imoveis.columns:
    corr_with_target = df_imoveis.corr()['Fire Alarm'].abs().sort_values(ascending=False)
    display(corr_with_target)
else:
    print("Coluna 'Fire Alarm' não encontrada no dataframe.")



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62630 entries, 0 to 62629
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      62630 non-null  int64  
 1   UTC             62630 non-null  int64  
 2   Temperature[C]  62630 non-null  float64
 3   Humidity[%]     62630 non-null  float64
 4   TVOC[ppb]       62630 non-null  int64  
 5   eCO2[ppm]       62630 non-null  int64  
 6   Raw H2          62630 non-null  int64  
 7   Raw Ethanol     62630 non-null  int64  
 8   Pressure[hPa]   62630 non-null  float64
 9   PM1.0           62630 non-null  float64
 10  PM2.5           62630 non-null  float64
 11  NC0.5           62630 non-null  float64
 12  NC1.0           62630 non-null  float64
 13  NC2.5           62630 non-null  float64
 14  CNT             62630 non-null  int64  
 15  Fire Alarm      62630 non-null  int64  
dtypes: float64(8), int64(8)
memory usage: 7.6 MB
Unnamed: 0          int64
UTC  

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,62630.0,3.131450e+04,18079.868017,0.000000e+00,1.565725e+04,3.131450e+04,4.697175e+04,6.262900e+04
UTC,62630.0,1.654792e+09,110002.488078,1.654712e+09,1.654743e+09,1.654762e+09,1.654778e+09,1.655130e+09
Temperature[C],62630.0,1.597042e+01,14.359576,-2.201000e+01,1.099425e+01,2.013000e+01,2.540950e+01,5.993000e+01
Humidity[%],62630.0,4.853950e+01,8.865367,1.074000e+01,4.753000e+01,5.015000e+01,5.324000e+01,7.520000e+01
TVOC[ppb],62630.0,1.942058e+03,7811.589055,0.000000e+00,1.300000e+02,9.810000e+02,1.189000e+03,6.000000e+04
eCO2[ppm],62630.0,6.700210e+02,1905.885439,4.000000e+02,4.000000e+02,4.000000e+02,4.380000e+02,6.000000e+04
Raw H2,62630.0,1.294245e+04,272.464305,1.066800e+04,1.283000e+04,1.292400e+04,1.310900e+04,1.380300e+04
Raw Ethanol,62630.0,1.975426e+04,609.513156,1.531700e+04,1.943500e+04,1.950100e+04,2.007800e+04,2.141000e+04
Pressure[hPa],62630.0,9.386276e+02,1.331344,9.308520e+02,9.387000e+02,9.388160e+02,9.394180e+02,9.398610e+02
PM1.0,62630.0,1.005943e+02,922.524245,0.000000e+00,1.280000e+00,1.810000e+00,2.090000e+00,1.433369e+04


Contagem de valores negativos por coluna (se houver):
{'TVOC[ppb]': 0, 'eCO2[ppm]': 0, 'Raw H2': 0, 'Raw Ethanol': 0, 'Pressure[hPa]': 0, 'PM1.0': 0, 'PM2.5': 0, 'NC0.5': 0, 'NC1.0': 0, 'NC2.5': 0, 'CNT': 0}


Fire Alarm        1.000000
CNT               0.673762
Humidity[%]       0.399846
UTC               0.389404
datetime_sp       0.389404
datetime_utc      0.389404
Unnamed: 0        0.361351
Raw Ethanol       0.340652
Pressure[hPa]     0.249797
TVOC[ppb]         0.214743
Temperature[C]    0.163902
NC0.5             0.128118
PM1.0             0.110552
Raw H2            0.107007
eCO2[ppm]         0.097006
PM2.5             0.084916
NC1.0             0.082828
NC2.5             0.057707
Name: Fire Alarm, dtype: float64

# 2 - Para essa base, onde você realizará as previsões de fire alarm, qual modelo de machine learning você aplicará? Justifique.

* Regressão Logística 

Justificativa:

A escolha da Regressão Logística fundamenta-se, primeiramente, na natureza do problema, uma vez que a variável alvo (Fire Alarm) é binária. Por ser uma técnica desenvolvida especificamente para classificação dicotômica, ela oferece a robustez necessária para este cenário. Além disso, o modelo permite uma interpretação direta dos coeficientes, facilitando a compreensão da influência de cada sensor na detecção de incêndios — um fator crítico em sistemas de monitoramento e segurança.

Do ponto de vista computacional, o algoritmo demonstra alta eficiência para o volume de dados do projeto (cerca de 62 mil observações), apresentando baixo custo de memória e rápido treinamento. A possibilidade de aplicar regularização (L1/L2) também é estratégica para mitigar eventuais problemas de multicolinearidade entre os sensores e prevenir o overfitting. Por fim, a Regressão Logística estabelece um baseline sólido e de fácil integração com técnicas de validação cruzada, servindo como uma referência essencial de desempenho antes da exploração de modelos de maior complexidade.

# 3 - Separe a base em Y e X e já rode a instância do modelo que você utilizará.

In [21]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_SPLITS = 5
CLASS_WEIGHT = None  # ou 'balanced' se houver desbalanceamento forte

# 0) remover coluna índice 
if 'Unnamed: 0' in df_imoveis.columns:
    df_imoveis = df_imoveis.drop(columns=['Unnamed: 0'])

# 1) separar X e y
target_col = 'Fire Alarm'
X = df_imoveis.drop(columns=[target_col])
y = df_imoveis[target_col]

# 2) identificar colunas numéricas e não numéricas 
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = [c for c in X.columns if c not in numeric_cols]

print(f"Colunas numéricas usadas como features: {len(numeric_cols)}")
print(f"Colunas não-numéricas detectadas (serão descartadas a menos que convertidas): {non_numeric_cols}")

# 4) criar X_numeric (apenas colunas numéricas)
X_numeric = X[numeric_cols].copy()

# 5) split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X_numeric, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# 6) pipeline e validação cruzada estratificada
pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        solver='liblinear',
        random_state=RANDOM_STATE,
        max_iter=2000,
        class_weight=CLASS_WEIGHT
    )
)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Cross-validation (accuracy e f1)
cv_scores_acc = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='accuracy', n_jobs=-1, error_score='raise')
cv_scores_f1 = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='f1', n_jobs=-1, error_score='raise')

print(f"CV Accuracy ({N_SPLITS}-fold): mean={cv_scores_acc.mean():.4f}, std={cv_scores_acc.std():.4f}")
print(f"CV F1       ({N_SPLITS}-fold): mean={cv_scores_f1.mean():.4f}, std={cv_scores_f1.std():.4f}")

# 7) treinar no treino completo e avaliar no teste
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else None

print("\nTeste - Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report (teste):")
print(classification_report(y_test, y_pred, digits=4))

print("Confusion matrix (teste):")
print(confusion_matrix(y_test, y_pred))

# AUC (se pred_proba disponível)
if y_proba is not None:
    try:
        print("AUC-ROC (teste):", roc_auc_score(y_test, y_proba))
    except Exception:
        pass

Colunas numéricas usadas como features: 14
Colunas não-numéricas detectadas (serão descartadas a menos que convertidas): ['datetime_utc', 'datetime_sp']
CV Accuracy (5-fold): mean=0.9887, std=0.0011
CV F1       (5-fold): mean=0.9921, std=0.0007

Teste - Accuracy: 0.988743413699505

Classification report (teste):
              precision    recall  f1-score   support

           0     0.9767    0.9841    0.9804      3575
           1     0.9936    0.9906    0.9921      8951

    accuracy                         0.9887     12526
   macro avg     0.9851    0.9873    0.9862     12526
weighted avg     0.9888    0.9887    0.9888     12526

Confusion matrix (teste):
[[3518   57]
 [  84 8867]]
AUC-ROC (teste): 0.9987629307347775


# 4 - Defina o número de Folds e rode o modelo com a validação cruzada.

In [29]:
RANDOM_STATE = 42
N_SPLITS = 5            
CLASS_WEIGHT = None

# 
if 'Unnamed: 0' in df_imoveis.columns:
    df_imoveis = df_imoveis.drop(columns=['Unnamed: 0'])

target_col = 'Fire Alarm'
X = df_imoveis.drop(columns=[target_col])
y = df_imoveis[target_col]

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = [c for c in X.columns if c not in numeric_cols]
print("Colunas numéricas usadas:", len(numeric_cols))
print("Colunas não numéricas descartadas:", non_numeric_cols)

X_numeric = X[numeric_cols].copy()

# 1) Pipeline (scaler + logistic)
pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver='liblinear', random_state=RANDOM_STATE,
                       max_iter=2000, class_weight=CLASS_WEIGHT)
)

# 2) Estratified K-Fold
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# 3) Cross-validate com múltiplas métricas
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_res = cross_validate(pipe, X_numeric, y, cv=skf, scoring=scoring, n_jobs=-1,
                        return_train_score=False, error_score='raise')

# 4) Mostrar resultados por métrica (mean + std) e por fold
import numpy as np
print(f"\nValidação cruzada ({N_SPLITS}-fold) — médias e desvios:")
for metric in scoring:
    scores = cv_res[f'test_{metric}']
    print(f"{metric:8s}: mean={scores.mean():.4f}, std={scores.std():.4f}, per-fold={np.round(scores,4)}")

# 5) Previsões cross-validated (para matriz de confusão/relatório agregado)
y_pred_cv = cross_val_predict(pipe, X_numeric, y, cv=skf, method='predict', n_jobs=-1)
print("\nClassification report (agregado via cross_val_predict):")
print(classification_report(y, y_pred_cv, digits=4))

print("Confusion matrix (agregado):")
print(confusion_matrix(y, y_pred_cv))


Colunas numéricas usadas: 14
Colunas não numéricas descartadas: ['datetime_utc', 'datetime_sp']

Validação cruzada (5-fold) — médias e desvios:
accuracy: mean=0.9896, std=0.0006, per-fold=[0.9888 0.989  0.99   0.9898 0.9905]
precision: mean=0.9933, std=0.0002, per-fold=[0.9931 0.9934 0.9932 0.9936 0.9934]
recall  : mean=0.9921, std=0.0008, per-fold=[0.9913 0.9912 0.9928 0.9921 0.9933]
f1      : mean=0.9927, std=0.0004, per-fold=[0.9922 0.9923 0.993  0.9928 0.9934]
roc_auc : mean=0.9990, std=0.0002, per-fold=[0.9991 0.9991 0.9987 0.9991 0.999 ]

Classification report (agregado via cross_val_predict):
              precision    recall  f1-score   support

           0     0.9804    0.9833    0.9818     17873
           1     0.9933    0.9921    0.9927     44757

    accuracy                         0.9896     62630
   macro avg     0.9868    0.9877    0.9873     62630
weighted avg     0.9896    0.9896    0.9896     62630

Confusion matrix (agregado):
[[17575   298]
 [  352 44405]]


# 5 - Avalie a pontuação de cada modelo e ao final a validação final da média.

Durante a validação cruzada estratificada 5-fold o pipeline (StandardScaler + LogisticRegression) apresentou desempenho muito elevado e estável: accuracy = 0.98962 (std = 0.00063), precision = 0.99334 (std = 0.00017), recall = 0.99214 (std = 0.00082), F1 = 0.99274 (std = 0.00045) e ROC AUC = 0.99900 (std = 0.00015) — valores que foram recalculados a partir dos scores por fold fornecidos e conferem com os números impressos no notebook (diferenças meramente por arredondamento). Em termos práticos, o modelo demonstra excelente capacidade de discriminação e baixa variabilidade entre folds, porém métricas tão altas recomendam uma checagem adicional para descartar data leakage, dependências temporais entre amostras ou features que contenham informação direta do alvo antes de considerar o modelo para produção.